In [1]:
#setup notebook
%load_ext autoreload
%autoreload 2
%matplotlib inline
#%matplotlib notebook
#%autosave 0
import base64
import pandas as pd
import numpy as np
import uuid
import os
import sys
import logging
logging.basicConfig(format='%(asctime)s %(levelname)s:%(message)s', level=logging.INFO, datefmt='%I:%M:%S')

import sdata
from sdata import SUUID
from sdata.base import Base, sdata_factory
from sdata.sclass import Blob

In [2]:
b = Blob(name="myblob")
b.mdf

,name,value,unit,dtype,description,label,required,ontology
key,,,,,,,,
_sdata_class,_sdata_class,sdata.base.Base:Blob,-,str,sdata class,,True,
_sdata_ctime,_sdata_ctime,2025-10-16T13:03:19.664184+00:00,-,str,UTC creation date,,False,
_sdata_name,_sdata_name,myblob,-,str,name of the data object,,True,
_sdata_parent_sname,_sdata_parent_sname,,-,str,sname of the parent,,False,
_sdata_project_sname,_sdata_project_sname,,-,str,sname of the project,,False,
_sdata_sname,_sdata_sname,Blob__myblob__b0c6ae852d7f4fca9f8ff1d23ee65a2e,-,str,sname of the data object,,True,
_sdata_topology_class,_sdata_topology_class,sdata.sclass:IndependentContinuant,-,str,sdata topology class name,,False,
_sdata_version,_sdata_version,1.0.0,-,str,sdata package version,,True,
checksum,checksum,None,None,str,A hash value (based on the SHA-256 algorithm) ...,Checksum (SHA-256),True,saf:checksum


In [15]:

import logging
import io
import base64
import hashlib
import os
from typing import Any, Dict, Optional, Literal
# Konfiguriere Logging für Debug-Ausgaben
logging.basicConfig(level=logging.DEBUG)

# Beispiel 1: Blob mit In-Memory-Bytes (z. B. für ein kleines PDF oder Bild)
example_bytes = b'%PDF-1.4\n% Beispiel-PDF-Inhalt'  # Platzhalter für echte Bytes, z. B. aus open('file.pdf', 'rb').read()
blob_bytes = Blob(
    content_type='bytes',
    value=example_bytes,
    filetype='pdf',
    name="MeinPDFBlob",
    description="Ein Blob mit PDF-Inhalt als Bytes."
)

# Zugriff auf Metadaten (ohne Lazy-Loading des Inhalts)
print("Blob-Name:", blob_bytes.name)
print("Filetype:", blob_bytes.filetype)
print("Exists:", blob_bytes.exists())  # True, da In-Memory

# Lazy-Loading: Inhalt erst jetzt laden (hier schon in base64 gespeichert)
content = blob_bytes.content_bytes
print("Inhalt als Bytes (Auszug):", content[:20])  # Zeigt die ersten 20 Bytes

# Hash-Berechnung (lädt Inhalt lazy)
print("SHA1:", blob_bytes.sha1)
print("MD5:", blob_bytes.md5)

# Serialisieren zu Dict/JSON
serialized = blob_bytes.to_dict()
print("Serialisierte Daten:", serialized)

# Deserialisieren zurück
restored_blob = Blob.from_dict(serialized)
print("Restored Filetype:", restored_blob.filetype)
print("Restored SHA1:", restored_blob.sha1)  # Lädt Inhalt lazy für Hash

# Beispiel 2: Blob mit URI (z. B. lokaler Dateipfad)
# Angenommen, es gibt eine Datei '/tmp/example.png'
with open('/tmp/example.png', 'wb') as f:
    f.write(b'\x89PNG\r\n\x1a\n')  # Platzhalter für PNG-Header

blob_uri = Blob(
    content_type='uri',
    value='/tmp/example.png',  # Oder 's3://mybucket/key.jpg' für S3
    filetype='png',
    name="MeinBildBlob",
    description="Ein Blob mit URI zu einem PNG-Bild."
)

# Metadaten-Zugriff (kein Loading)
print("Blob-Name:", blob_uri.name)
print("Filetype:", blob_uri.filetype)
print("Exists:", blob_uri.exists())  # Überprüft via fsspec, ohne Inhalt zu laden

# Lazy-Loading: Inhalt laden
content_uri = blob_uri.content_bytes
print("Inhalt als Bytes (Auszug):", content_uri[:20])

# Hash-Berechnung
print("SHA1:", blob_uri.sha1)
print("MD5:", blob_uri.md5)

# Beispiel 3: Blob mit URI in Zip (fsspec unterstützt das)
# Angenommen, '/tmp/outer.zip' enthält 'inner.pdf'
blob_zip = Blob(
    content_type='uri',
    value='zip://inner.pdf::/tmp/outer.zip',
    filetype='pdf',
    name="MeinZipBlob"
)

if blob_zip.exists():
    print("Zip-Inhalt exists und kann lazy geladen werden.")

# Aktualisieren des Inhalts
blob_bytes.set_content(content_type='uri', value='/tmp/newfile.txt', filetype='txt')
print("Updated Filetype:", blob_bytes.filetype)

Blob-Name: MeinPDFBlob


KeyError: 'content'

In [ ]:
# Konfiguriere Logging für Debug-Ausgaben
logging.basicConfig(level=logging.DEBUG)

# Beispiel 1: Blob mit In-Memory-Bytes (z. B. für ein kleines PDF oder Bild)
example_bytes = b'%PDF-1.4\n% Beispiel-PDF-Inhalt'  # Platzhalter für echte Bytes, z. B. aus open('file.pdf', 'rb').read()
blob_bytes = Blob(
    content_type='bytes',
    value=example_bytes,
    filetype='pdf',
    name="MeinPDFBlob",
    description="Ein Blob mit PDF-Inhalt als Bytes."
)

# Zugriff auf Metadaten (ohne Lazy-Loading des Inhalts)
print("Blob-Name:", blob_bytes.metadata.attributes['_sdata_name'].value)
print("Filetype:", blob_bytes.filetype)
print("Exists:", blob_bytes.exists())  # True, da In-Memory

# Lazy-Loading: Inhalt erst jetzt laden (hier schon in base64 gespeichert)
content = blob_bytes.content_bytes
print("Inhalt als Bytes (Auszug):", content[:20])  # Zeigt die ersten 20 Bytes

# Hash-Berechnung (lädt Inhalt lazy)
print("SHA1:", blob_bytes.sha1)
print("MD5:", blob_bytes.md5)

# Serialisieren zu Dict/JSON
serialized = blob_bytes.to_dict()
print("Serialisierte Daten:", serialized)

# Deserialisieren zurück (jetzt fehlerfrei)
restored_blob = Blob.from_dict(serialized)
print("Restored Filetype:", restored_blob.filetype)
print("Restored SHA1:", restored_blob.sha1)  # Lädt Inhalt lazy für Hash

In [ ]:
ls /tmp/example.png

In [ ]:
# Beispiel 2: Blob mit URI (z. B. lokaler Dateipfad)
# Erzeuge eine Testdatei
test_file_path = '/tmp/example.png'
with open(test_file_path, 'wb') as f:
    f.write(b'\x89PNG\r\n\x1a\n')  # Platzhalter für PNG-Header

blob_uri = Blob(
    content_type='uri',
    value=test_file_path,  # Oder 's3://mybucket/key.jpg' für S3 (benötigt Credentials)
    filetype='png',
    name="MeinBildBlob",
    description="Ein Blob mit URI zu einem PNG-Bild."
)

# Metadaten-Zugriff (kein Loading)
print("Blob-Name:", blob_uri.metadata.attributes['_sdata_name'].value)
print("Filetype:", blob_uri.filetype)
print("Exists:", blob_uri.exists())  # Überprüft via fsspec, ohne Inhalt zu laden (True, wenn Datei existiert)

# Lazy-Loading: Inhalt laden
content_uri = blob_uri.content_bytes
print("Inhalt als Bytes (Auszug):", content_uri[:20])

# Hash-Berechnung
print("SHA1:", blob_uri.sha1)
print("MD5:", blob_uri.md5)

# Aktualisieren des Inhalts
blob_bytes.set_content(content_type='uri', value=test_file_path, filetype='png')
print("Updated Filetype:", blob_bytes.filetype)

In [ ]:
os.path.exists(test_file_path)

In [ ]:
b=Blob()
b

In [ ]:
b.data

In [ ]:
b.content_bytes